# qcnn baseline

our other circuits read all 8 qubits, which makes them act like an additive model. the qcnn pools 8 qubits to 4 to 2 and reads only the last 2, so the other 6 can only reach the readout through two-qubit gates.

- ry encoding, features to qubits in ascending component order
- convolution: the general two-qubit block (vatan and williams, 15 angles) on even pairs, then odd pairs
- pooling: crz and crx from the dropped qubit onto its neighbour
- 8 -> 4 -> 2, a last convolution, <z> on 2 qubits, weighted sum and bias
- weights shared within a layer: 3 x 15 + 2 x 2 + 3 = 52

runs on the best 8 and pls 8 with 3 seeds, next to the sin/cos twin and the additive circuit.

attach the prep and clip features datasets and turn internet on (pennylane install). cpu, runs on all cores, about 25 minutes.

In [1]:
!pip install -q pennylane==0.45.1

import os
os.environ["OMP_NUM_THREADS"] = "1"   # one thread per process, set before numpy and torch load, about 2x faster
import time
import multiprocessing as mp
import numpy as np
import pandas as pd
import torch
import pennylane as qml
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

seed = 0
np.random.seed(seed)
torch.set_num_threads(1)

prep_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-feature-dataset/prep"
clip_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-clip-dataset/features"
out_dir = "/kaggle/working/qcnn"
os.makedirs(out_dir, exist_ok=True)

seeds = [0, 1, 2]
lr, epochs, batch_size = 0.01, 5, 256
c_grid = [0.01, 0.1, 1.0]

In [2]:
angles, meta = {}, {}
for split in ["train", "val"]:
    angles[split] = np.load(f"{prep_dir}/ffpp_{split}_angles32.npy").astype(np.float64)
    meta[split] = pd.read_csv(f"{prep_dir}/ffpp_{split}_rows.csv")

is_inner = np.load(f"{prep_dir}/ffpp_train_is_inner.npy").astype(bool)
comp = pd.read_csv(f"{prep_dir}/per_component_auc.csv")
best8 = sorted(comp.sort_values("strength_search_train", ascending=False)["component"].head(8).tolist())
first8 = list(range(8))

y_train = meta["train"]["label"].to_numpy()
y_val = meta["val"]["label"].to_numpy()
# a video is (method, video_id), the 4 fakes of one pair share a video_id
val_video = meta["val"].groupby(["method", "video_id"]).ngroup().to_numpy()

print("train frames:", len(y_train), " val frames:", len(y_val), " val videos:", val_video.max() + 1)
print("best 8:", best8)

train frames: 46076  val frames: 22396  val videos: 700
best 8: [2, 3, 4, 8, 10, 16, 17, 21]


## the two inputs

best 8 from the per-component auc, and 8 pls components fitted on search-train only and scaled with search-train percentiles.

In [3]:
clip = {}
for split in ["train", "val"]:
    clip_rows = pd.read_csv(f"{clip_dir}/ffpp_{split}_rows.csv")
    assert (clip_rows["path"].to_numpy() == meta[split]["path"].to_numpy()).all(), split
    x = np.load(f"{clip_dir}/ffpp_{split}_clip.npy").astype(np.float64)
    clip[split] = x / np.linalg.norm(x, axis=1, keepdims=True)
st = ~is_inner

pls = PLSRegression(n_components=8, scale=False)
pls.fit(clip["train"][st], y_train[st])
z = {split: pls.transform(clip[split]) for split in clip}
lo = np.percentile(z["train"][st], 0.5, axis=0)
hi = np.percentile(z["train"][st], 99.5, axis=0)

inputs = {
    "best 8": {split: angles[split][:, best8] for split in angles},
    "PLS 8": {split: np.clip((z[split] - lo) / (hi - lo), 0.0, 1.0) * np.pi for split in z},
}
for name in inputs:
    print(name, inputs[name]["train"].shape, inputs[name]["val"].shape)

def video_level(prob):
    df = pd.DataFrame({"video": val_video, "p": prob, "y": y_val})
    g = df.groupby("video").agg(p=("p", "mean"), y=("y", "first"))
    return g["p"].to_numpy(), g["y"].to_numpy()


def bootstrap_ci(p, y, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    aucs = []
    while len(aucs) < n:
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aucs, [2.5, 97.5])


def score(prob):
    pv, yv = video_level(prob)
    lo, hi = bootstrap_ci(pv, yv)
    return {"frame auc": roc_auc_score(y_val, prob), "video auc": roc_auc_score(yv, pv), "ci low": lo, "ci high": hi}


def sincos(a):
    return np.concatenate([np.sin(a), np.cos(a)], axis=1)


def predict(model, x, batch_size=4096):
    out = []
    with torch.no_grad():
        for s in range(0, len(x), batch_size):
            xb = torch.as_tensor(x[s:s + batch_size], dtype=torch.float64)
            out.append(torch.sigmoid(model(xb)))
    return torch.cat(out).numpy()

best 8 (46076, 8) (22396, 8)
PLS 8 (46076, 8) (22396, 8)


## the qcnn

In [4]:
n_qubits = 8


def conv_block(w, a, b):
    qml.Rot(w[0], w[1], w[2], wires=a)
    qml.Rot(w[3], w[4], w[5], wires=b)
    qml.CNOT(wires=[b, a])
    qml.RZ(w[6], wires=a)
    qml.RY(w[7], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(w[8], wires=b)
    qml.CNOT(wires=[b, a])
    qml.Rot(w[9], w[10], w[11], wires=a)
    qml.Rot(w[12], w[13], w[14], wires=b)


def pool_block(w, drop, keep):
    qml.CRZ(w[0], wires=[drop, keep])
    qml.PauliX(wires=drop)
    qml.CRX(w[1], wires=[drop, keep])


def conv_layer(w, active):
    n = len(active)
    if n == 2:
        conv_block(w, active[0], active[1])
        return
    for i in range(0, n, 2):
        conv_block(w, active[i], active[i + 1])
    for i in range(1, n, 2):
        conv_block(w, active[i], active[(i + 1) % n])


def pool_layer(w, active):
    for i in range(0, len(active), 2):
        pool_block(w, active[i + 1], active[i])
    return active[::2]


dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def qcnn_circuit(x, conv_w, pool_w):
    for i in range(n_qubits):
        qml.RY(x[:, i], wires=i)
    active = list(range(n_qubits))
    for layer in range(2):
        conv_layer(conv_w[layer], active)
        active = pool_layer(pool_w[layer], active)
    conv_layer(conv_w[2], active)
    return [qml.expval(qml.PauliZ(q)) for q in active]


@qml.qnode(dev, interface="torch", diff_method="backprop")
def additive_circuit(x, rot_w):
    for i in range(n_qubits):
        qml.RY(x[:, i], wires=i)
        qml.Rot(rot_w[i, 0], rot_w[i, 1], rot_w[i, 2], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class QCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_w = torch.nn.Parameter(0.1 * torch.randn(3, 15, dtype=torch.float64))
        self.pool_w = torch.nn.Parameter(0.1 * torch.randn(2, 2, dtype=torch.float64))
        self.readout = torch.nn.Linear(2, 1, dtype=torch.float64)
        torch.nn.init.normal_(self.readout.weight, std=0.1)
        torch.nn.init.zeros_(self.readout.bias)

    def forward(self, x):
        return self.readout(torch.stack(qcnn_circuit(x, self.conv_w, self.pool_w), dim=1)).squeeze(1)


class Additive(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rot_w = torch.nn.Parameter(0.1 * torch.randn(n_qubits, 3, dtype=torch.float64))
        self.readout = torch.nn.Linear(n_qubits, 1, dtype=torch.float64)
        torch.nn.init.normal_(self.readout.weight, std=0.1)
        torch.nn.init.zeros_(self.readout.bias)

    def forward(self, x):
        return self.readout(torch.stack(additive_circuit(x, self.rot_w), dim=1)).squeeze(1)


models = {"qcnn": QCNN, "additive circuit": Additive}


def train(model, x, y, lr=0.01, epochs=5, batch_size=256, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.as_tensor(x, dtype=torch.float64)
    y = torch.as_tensor(y, dtype=torch.float64)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        order = torch.randperm(len(x), generator=g)
        for start in range(0, len(x), batch_size):
            idx = order[start:start + batch_size]
            opt.zero_grad()
            loss_fn(model(x[idx]), y[idx]).backward()
            opt.step()


n_weights = {k: sum(p.numel() for p in m().parameters()) for k, m in models.items()}
print("trainable weights:", n_weights, "(expected 52 for the qcnn)")

trainable weights: {'qcnn': 52, 'additive circuit': 33} (expected 52 for the qcnn)


## does it combine features

two toy tasks on features 0 and 1, which the qcnn combines in its first layer. it should learn both, the additive circuit only the first.

In [5]:
rng = np.random.default_rng(0)
toy_x = rng.uniform(0, np.pi, (2000, n_qubits))
toy_tasks = {"additive": (toy_x[:, 0] + toy_x[:, 1] > np.pi).astype(float),
             "interaction": (np.cos(toy_x[:, 0]) * np.cos(toy_x[:, 1]) > 0).astype(float)}

for task, labels in toy_tasks.items():
    for kind in models:
        torch.manual_seed(0)
        model = models[kind]()
        train(model, toy_x[:1000], labels[:1000], lr=0.05, epochs=10, batch_size=100, seed=0)
        print(f"{task:12s} {kind:17s} test auc {roc_auc_score(labels[1000:], predict(model, toy_x[1000:])):.3f}")

additive     qcnn              test auc 0.973
additive     additive circuit  test auc 0.995
interaction  qcnn              test auc 0.977
interaction  additive circuit  test auc 0.505


## sin/cos twin on both inputs

In [6]:
table = []
for name, data in inputs.items():
    best = None
    for c in c_grid:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(sincos(data["train"]), y_train)
        row = score(model.predict_proba(sincos(data["val"]))[:, 1])
        if best is None or row["video auc"] > best["video auc"]:
            best = dict(row, input=name, model="sin/cos twin", seed=-1, weights=17, minutes=0.0)
    table.append(best)

## qcnn and additive circuit in parallel

In [7]:
def run_task(task):
    name, kind, s = task
    t0 = time.time()
    torch.manual_seed(s)
    model = models[kind]()
    train(model, inputs[name]["train"], y_train, lr=lr, epochs=epochs, batch_size=batch_size, seed=s)
    row = {"input": name, "model": kind, "seed": s, "weights": n_weights[kind]}
    row.update(score(predict(model, inputs[name]["val"])))
    row["minutes"] = (time.time() - t0) / 60
    return row


tasks = [(i, "qcnn", s) for i in inputs for s in seeds] + [(i, "additive circuit", 0) for i in inputs]
with mp.get_context("fork").Pool(os.cpu_count()) as pool:
    table += pool.map(run_task, tasks)

## results

differences under about 0.02 are inside the noise of one val run.

In [8]:
res = pd.DataFrame(table)
res.to_csv(f"{out_dir}/qcnn_runs.csv", index=False)
summary = res.groupby(["input", "model"]).agg(weights=("weights", "first"), video_auc_mean=("video auc", "mean"),
                                              video_auc_std=("video auc", "std"), ci_low_worst=("ci low", "min"),
                                              ci_high_best=("ci high", "max"), minutes=("minutes", "mean")).reset_index()
summary.to_csv(f"{out_dir}/qcnn_summary.csv", index=False)
print(summary.round(3).to_string(index=False))

for name in inputs:
    s = summary[summary["input"] == name].set_index("model")["video_auc_mean"]
    print(f"{name}: qcnn minus twin {s['qcnn'] - s['sin/cos twin']:+.3f}, qcnn minus additive {s['qcnn'] - s['additive circuit']:+.3f}")

 input            model  weights  video_auc_mean  video_auc_std  ci_low_worst  ci_high_best  minutes
 PLS 8 additive circuit       33           0.841            NaN         0.808         0.872    0.998
 PLS 8             qcnn       52           0.841          0.002         0.807         0.873    7.070
 PLS 8     sin/cos twin       17           0.842            NaN         0.809         0.872    0.000
best 8 additive circuit       33           0.712            NaN         0.667         0.753    1.033
best 8             qcnn       52           0.702          0.007         0.650         0.748    9.635
best 8     sin/cos twin       17           0.713            NaN         0.667         0.754    0.000
best 8: qcnn minus twin -0.010, qcnn minus additive -0.010
PLS 8: qcnn minus twin -0.001, qcnn minus additive +0.000
